# 2장 1강: t-검정의 이론과 가정 — 실습문제

## 실습 목표

- 평균 차이와 표준오차를 이용해 t통계량을 직접 계산할 수 있다.
- 단일표본 t검정으로 하나의 표본평균과 기준값을 비교할 수 있다.
- 독립성·정규성·등분산성을 확인한 뒤 독립표본 t검정을 수행할 수 있다.
- 독립표본과 대응표본 상황을 구분하고 적절한 함수를 선택할 수 있다.
- p-value를 유의수준과 비교하여 검정 결과를 올바르게 해석할 수 있다.

## 실습 환경 / 데이터

- Python
- NumPy, pandas
- scipy.stats
- `ames_housing.csv`

| 컬럼 | 의미 |
|---|---|
| `SalePrice` | 주택 판매가격 |
| `KitchenQual` | 주방 품질 |

> 모든 검정은 양측검정이며 유의수준 `α = 0.05`를 사용합니다.  
> Ames 표본 추출에는 지정된 `random_state`를 사용해 결과를 재현합니다.

## 실습 준비

1. 필요한 라이브러리를 불러오세요.
2. Ames Housing 데이터를 `df`에 불러오세요.
3. 데이터 크기, 전체 결측치 수, 컬럼명과 상위 5개 행을 확인하세요.

In [1]:
# [실습 준비] 라이브러리 임포트 및 데이터 불러오기
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

# 1. 데이터 경로 설정
ROOT = Path.cwd()
data_path = ROOT / "ames_housing.csv"
if not data_path.exists():
    data_path = ROOT / "data" / "ames_housing.csv"

# 2. 데이터 불러오기
df = pd.read_csv(data_path, encoding="utf-8-sig")

# 3. 데이터 크기, 결측치, 컬럼명, 상위 5행 확인
print("=" * 60)
print("[데이터 기본 정보]")
print(f"- 데이터 형상 (Rows, Cols): {df.shape}")
print(f"- 전체 결측치 수: {df.isnull().sum().sum():,}건")
print(f"- 컬럼명: {list(df.columns)}")
print("=" * 60)
df.head()

[데이터 기본 정보]
- 데이터 형상 (Rows, Cols): (1460, 10)
- 전체 결측치 수: 0건
- 컬럼명: ['SalePrice', 'GrLivArea', 'LotArea', 'OverallQual', 'KitchenQual', 'CentralAir', 'HeatingQC', 'PavedDrive', 'Neighborhood', 'YearBuilt']


,SalePrice,GrLivArea,LotArea,OverallQual,KitchenQual,CentralAir,HeatingQC,PavedDrive,Neighborhood,YearBuilt
0,208500,1710,8450,7,Gd,Y,Ex,Y,CollgCr,2003
1,181500,1262,9600,6,TA,Y,Ex,Y,Veenker,1976
2,223500,1786,11250,7,Gd,Y,Ex,Y,CollgCr,2001
3,140000,1717,9550,7,Gd,Y,Gd,Y,Crawfor,1915
4,250000,2198,14260,8,Gd,Y,Ex,Y,NoRidge,2000


---

## 필수 1. t통계량 직접 계산과 단일표본 t검정

### 문제 1-1. 평균 판매가격은 180,000달러와 다른가?

#### 문제 설명

Ames 주택 30개를 표본으로 추출하여 평균 판매가격이 기준값 180,000달러와 다른지 확인합니다. 먼저 t통계량을 직접 계산한 뒤 `ttest_1samp()` 결과와 비교하세요.

#### 요구사항

1. `SalePrice`에서 `n=30`, `random_state=2`로 표본을 추출하여 `sale_sample`에 저장하세요.
2. 표본 수, 표본평균, 표본표준편차를 출력하세요.
3. `stats.sem()`을 이용해 표준오차를 계산하세요.
4. 다음 식으로 t통계량을 직접 계산하세요.  
   `t = (표본평균 - 기준값) / 표준오차`
5. Shapiro-Wilk 검정으로 표본의 정규성을 확인하세요.
6. 다음 가설을 작성하세요.
   - H₀: 모집단 평균 판매가격은 180,000달러이다.
   - H₁: 모집단 평균 판매가격은 180,000달러가 아니다.
7. `stats.ttest_1samp()`로 단일표본 t검정을 수행하세요.
8. 직접 계산한 t통계량과 함수가 반환한 t통계량을 비교하세요.
9. p-value를 이용해 귀무가설 기각 여부를 판단하세요.

#### 해석 질문

**Q1.** t통계량은 평균 차이와 표준오차를 어떻게 이용한 값인가요?  
**Q2.** 같은 평균 차이라면 표준오차가 작아질수록 t통계량의 절댓값은 어떻게 변하나요?  
**Q3.** 직접 계산한 t통계량과 `ttest_1samp()`의 t통계량은 일치하나요?  
**Q4.** 검정 결과 평균 판매가격이 180,000달러와 다르다고 판단할 수 있나요?

#### 제출 결과

- 기술통계량과 정규성 결과
- 직접 계산한 표준오차와 t통계량
- 단일표본 t검정 결과
- 귀무가설 판단과 해석
- Q1~Q4 답변

In [2]:
# [필수 1] t통계량 직접 계산과 단일표본 t검정

# 1. n=30, random_state=2로 표본 추출
sale_sample = df["SalePrice"].sample(n=30, random_state=2)

# 2. 표본 수, 표본평균, 표본표준편차
n = len(sale_sample)
mean_val = sale_sample.mean()
std_val = sale_sample.std(ddof=1)

# 3. 표준오차 계산
sem_val = stats.sem(sale_sample)

# 4. t통계량 직접 계산: t = (표본평균 - 기준값) / 표준오차
mu0 = 180000
t_manual = (mean_val - mu0) / sem_val

# 5. Shapiro-Wilk 정규성 검정
shapiro_stat, shapiro_p = stats.shapiro(sale_sample)

# 6. 가설
# H0: 모집단 평균 판매가격은 180,000달러이다. (mu = 180000)
# H1: 모집단 평균 판매가격은 180,000달러가 아니다. (mu != 180000)

# 7. 단일표본 t검정 수행
t_func, p_func = stats.ttest_1samp(sale_sample, popmean=mu0, alternative="two-sided")

# 9. 유의수준과 비교
alpha = 0.05

print("=" * 60)
print("[필수 1: 단일표본 t검정 결과]")
print("=" * 60)
print(f"1. 표본 수: {n}개 | 표본평균: ${mean_val:,.2f} | 표본표준편차: ${std_val:,.2f}")
print(f"2. 표준오차(SEM): {sem_val:,.4f}")
print(f"3. 직접 계산한 t통계량: t = ({mean_val:,.2f} - {mu0:,}) / {sem_val:,.4f} = {t_manual:.4f}")
print(
    f"4. Shapiro-Wilk 정규성 검정: W = {shapiro_stat:.4f}, p = {shapiro_p:.4f} -> "
    f"{'정규성 기각 불가' if shapiro_p >= 0.05 else '정규성 기각'}"
)
print("-" * 60)
print(f"5. stats.ttest_1samp() 결과: t = {t_func:.4f}, p-value = {p_func:.4f}")
print(
    f"6. 직접 계산 t({t_manual:.4f})와 함수 t({t_func:.4f}) 비교: "
    f"차이 = {abs(t_manual - t_func):.10f} -> {'일치' if abs(t_manual - t_func) < 1e-6 else '불일치'}"
)
print("-" * 60)
if p_func < alpha:
    print(f"판단: p-value({p_func:.4f}) < {alpha} 이므로 귀무가설을 기각합니다.")
    print("결론: 평균 판매가격은 180,000달러와 통계적으로 유의하게 다릅니다.")
else:
    print(f"판단: p-value({p_func:.4f}) >= {alpha} 이므로 귀무가설을 기각할 수 없습니다.")
    print("결론: 평균 판매가격이 180,000달러와 다르다고 볼 통계적 증거가 부족합니다.")
print("=" * 60)

[필수 1: 단일표본 t검정 결과]
1. 표본 수: 30개 | 표본평균: $233,323.80 | 표본표준편차: $86,013.01
2. 표준오차(SEM): 15,703.7556
3. 직접 계산한 t통계량: t = (233,323.80 - 180,000) / 15,703.7556 = 3.3956
4. Shapiro-Wilk 정규성 검정: W = 0.9538, p = 0.2133 -> 정규성 기각 불가
------------------------------------------------------------
5. stats.ttest_1samp() 결과: t = 3.3956, p-value = 0.0020
6. 직접 계산 t(3.3956)와 함수 t(3.3956) 비교: 차이 = 0.0000000000 -> 일치
------------------------------------------------------------
판단: p-value(0.0020) < 0.05 이므로 귀무가설을 기각합니다.
결론: 평균 판매가격은 180,000달러와 통계적으로 유의하게 다릅니다.


### 필수 1 답변 작성란

- **Q1. t통계량은 평균 차이와 표준오차를 어떻게 이용한 값인가요?**
  * **답변:** t통계량은 **(표본평균 - 기준값)을 표준오차로 나눈 값**입니다. 이번 표본에서는 t = (233,323.80 - 180,000) / 15,703.7556 = **3.3956**으로 계산되며, 평균 차이($53,323.80)를 표준오차 단위로 표준화하여 "평균 차이가 표본오차에 비해 얼마나 큰가"를 나타냅니다.
- **Q2. 같은 평균 차이라면 표준오차가 작아질수록 t통계량의 절댓값은 어떻게 변하나요?**
  * **답변:** 표준오차가 분모이므로 **표준오차가 작아질수록 |t|는 커집니다.** 표준오차가 더 작았다면 동일한 평균 차이에서도 t의 절댓값은 3.3956보다 더 커졌을 것입니다.
- **Q3. 직접 계산한 t통계량과 `ttest_1samp()`의 t통계량은 일치하나요?**
  * **답변:** **네, 일치합니다.** 직접 계산한 t = 3.3956과 `stats.ttest_1samp()`가 반환한 t = 3.3956은 부동소수점 오차 수준(차이 < 1e-6)에서 동일하며, 두 값은 실질적으로 같습니다.
- **Q4. 검정 결과 평균 판매가격이 180,000달러와 다르다고 판단할 수 있나요?**
  * **답변:** **네, 다르다고 판단할 수 있습니다.** p-value = 0.0020으로 유의수준 α = 0.05보다 작아 귀무가설(H0: μ = 180,000)을 기각하며, 표본평균이 $233,323.80으로 기준값보다 높게 나타난 것은 통계적으로 유의한 차이입니다.

---

## 필수 2. 독립표본 t검정의 가정 점검과 수행

### 문제 2-1. 주방 품질 `Gd`와 `TA` 집단의 평균 판매가격 비교

#### 문제 설명

주방 품질이 `Gd`인 주택과 `TA`인 주택에서 각각 20개를 추출해 평균 판매가격을 비교합니다. 두 집단은 서로 다른 주택으로 구성되어 있습니다.

#### 요구사항

1. `KitchenQual == "Gd"`와 `KitchenQual == "TA"` 집단의 `SalePrice`에서 각각 `n=20`, `random_state=42`로 표본을 추출하세요.
2. 두 집단의 표본 수와 평균을 확인하세요.
3. 데이터 수집 구조를 근거로 두 집단의 독립성을 설명하세요.
4. 각 집단에 Shapiro-Wilk 정규성 검정을 수행하세요.
5. 두 집단에 Levene 등분산 검정을 수행하세요.
6. 가정 점검 결과에 따라 `equal_var=True` 또는 `equal_var=False`를 결정하세요.
7. `stats.ttest_ind()`로 독립표본 t검정을 수행하세요.
8. t통계량과 p-value를 출력하고 두 집단 평균 차이를 해석하세요.

#### 해석 질문

**Q1.** 두 집단이 독립표본인 이유는 무엇인가요?  
**Q2.** 독립성은 별도의 p-value로 확인할 수 있나요?  
**Q3.** 정규성과 등분산성 가정은 각각 충족되나요?  
**Q4.** `equal_var`에는 어떤 값을 사용해야 하나요?  
**Q5.** 두 집단의 평균 판매가격에는 통계적으로 유의한 차이가 있나요?

#### 제출 결과

- 집단별 표본 수와 평균
- 독립성 설명
- 정규성 및 등분산성 검정 결과
- `equal_var` 선택 근거
- 독립표본 t검정 결과와 해석
- Q1~Q5 답변

In [3]:
# [필수 2] 독립표본 t검정의 가정 점검과 수행

# 1. Gd, TA 집단 각각 n=20, random_state=42로 표본 추출
group_gd = df[df["KitchenQual"] == "Gd"]["SalePrice"].sample(n=20, random_state=42)
group_ta = df[df["KitchenQual"] == "TA"]["SalePrice"].sample(n=20, random_state=42)

# 2. 표본 수와 평균 확인
n_gd, mean_gd = len(group_gd), group_gd.mean()
n_ta, mean_ta = len(group_ta), group_ta.mean()

print("=" * 65)
print("[필수 2: KitchenQual 'Gd' vs 'TA' 독립표본 t검정]")
print("=" * 65)
print(f"1. 표본 수 및 평균")
print(f"  - Gd: n = {n_gd}, 표본평균 = ${mean_gd:,.2f}")
print(f"  - TA: n = {n_ta}, 표본평균 = ${mean_ta:,.2f}")
print(f"  - 평균 차이(Gd - TA): ${mean_gd - mean_ta:,.2f}")
print("-" * 65)

# 3. 독립성 설명: 'Gd' 표본과 'TA' 표본은 서로 다른 주택들에서 각각 추출되었으므로
#    한 집단의 관측값이 다른 집단의 관측값에 영향을 주지 않는 독립표본이다.
print("2. 표본 관계: 서로 다른 주택으로 대응하지 않는 두 집단입니다.")
print("   관측치 간 독립성은 표집 구조를 바탕으로 가정하며 공간적 군집 여부는 추가 확인합니다.")
print("-" * 65)

# 4. 정규성 검정 (Shapiro-Wilk)
shapiro_gd = stats.shapiro(group_gd)
shapiro_ta = stats.shapiro(group_ta)

# 5. 등분산 검정 (Levene)
levene_result = stats.levene(group_gd, group_ta)

norm_gd = shapiro_gd.pvalue >= 0.05
norm_ta = shapiro_ta.pvalue >= 0.05
equal_var = levene_result.pvalue >= 0.05

print("3. 정규성 및 등분산성 검정 (alpha=0.05)")
print(
    f"  - Shapiro-Wilk (Gd): W = {shapiro_gd.statistic:.4f}, p = {shapiro_gd.pvalue:.4f} -> "
    f"{'정규성 기각 불가' if norm_gd else '정규성 기각'}"
)
print(
    f"  - Shapiro-Wilk (TA): W = {shapiro_ta.statistic:.4f}, p = {shapiro_ta.pvalue:.4f} -> "
    f"{'정규성 기각 불가' if norm_ta else '정규성 기각'}"
)
print(
    f"  - Levene 등분산성  : F = {levene_result.statistic:.4f}, p = {levene_result.pvalue:.4f} -> "
    f"{'등분산성 기각 불가' if equal_var else '등분산성 기각'}"
)
print("-" * 65)

# 6. 가정 점검 결과에 따라 equal_var 결정
print(f"4. equal_var 결정: Levene 검정 결과에 따라 equal_var={equal_var}를 사용합니다.")
print("-" * 65)

# 7. 독립표본 t검정 수행
t_stat, p_val = stats.ttest_ind(group_gd, group_ta, equal_var=equal_var)

# 8. 결과 출력 및 해석
alpha = 0.05
print(f"5. 독립표본 t검정 결과: t = {t_stat:.4f}, p-value = {p_val:.4f}")
if p_val < alpha:
    print(f"판단: p-value({p_val:.4f}) < {alpha} 이므로 귀무가설을 기각합니다.")
    print(
        f"결론: 주방 품질 'Gd'와 'TA' 주택의 평균 판매가격은 통계적으로 유의하게 다릅니다 "
        f"(Gd가 평균 ${mean_gd - mean_ta:,.2f} 더 높음)."
    )
else:
    print(f"판단: p-value({p_val:.4f}) >= {alpha} 이므로 귀무가설을 기각할 수 없습니다.")
print("=" * 65)

[필수 2: KitchenQual 'Gd' vs 'TA' 독립표본 t검정]
1. 표본 수 및 평균
  - Gd: n = 20, 표본평균 = $191,916.60
  - TA: n = 20, 표본평균 = $137,432.50
  - 평균 차이(Gd - TA): $54,484.10
-----------------------------------------------------------------
2. 표본 관계: 서로 다른 주택으로 대응하지 않는 두 집단입니다.
   관측치 간 독립성은 표집 구조를 바탕으로 가정하며 공간적 군집 여부는 추가 확인합니다.
-----------------------------------------------------------------
3. 정규성 및 등분산성 검정 (alpha=0.05)
  - Shapiro-Wilk (Gd): W = 0.9555, p = 0.4580 -> 정규성 기각 불가
  - Shapiro-Wilk (TA): W = 0.9410, p = 0.2503 -> 정규성 기각 불가
  - Levene 등분산성  : F = 3.7806, p = 0.0593 -> 등분산성 기각 불가
-----------------------------------------------------------------
4. equal_var 결정: Levene 검정 결과에 따라 equal_var=True를 사용합니다.
-----------------------------------------------------------------
5. 독립표본 t검정 결과: t = 3.4249, p-value = 0.0015
판단: p-value(0.0015) < 0.05 이므로 귀무가설을 기각합니다.
결론: 주방 품질 'Gd'와 'TA' 주택의 평균 판매가격은 통계적으로 유의하게 다릅니다 (Gd가 평균 $54,484.10 더 높음).


### 필수 2 답변 작성란

- **Q1. 두 집단이 독립표본인 이유는 무엇인가요?**
  * **답변:** `KitchenQual == "Gd"` 표본(n=20)과 `KitchenQual == "TA"` 표본(n=20)은 **서로 다른 주택들**에서 각각 추출되었으며, 한 표본의 관측값이 다른 표본의 관측값에 어떤 영향도 주지 않으므로 독립표본입니다.
- **Q2. 독립성은 별도의 p-value로 확인할 수 있나요?**
  * **답변:** **아니요.** 독립성은 정규성(Shapiro-Wilk)이나 등분산성(Levene)처럼 통계 검정으로 산출되는 값이 아니라, **데이터가 어떻게 수집되었는지(설계 구조)**를 근거로 판단하는 것입니다. 이 문제에서는 서로 다른 주택 집단에서 추출했다는 사실 자체가 독립성의 근거입니다.
- **Q3. 정규성과 등분산성 가정은 각각 충족되나요?**
  * **답변:** **둘 다 충족됩니다.** Shapiro-Wilk 검정 결과 Gd(p = 0.4580)와 TA(p = 0.2503) 모두 p ≥ 0.05로 정규성을 만족합니다. Levene 검정 결과는 p = 0.0593으로 0.05보다 약간 크므로 등분산 가정도 기각되지 않아 충족되는 것으로 판단하지만, 경계값에 가까운 수치라는 점은 유의할 필요가 있습니다.
- **Q4. `equal_var`에는 어떤 값을 사용해야 하나요?**
  * **답변:** 정규성과 등분산성이 모두 충족되었으므로 **`equal_var=True`**를 사용하여 전통적인 Student's t-검정을 수행합니다.
- **Q5. 두 집단의 평균 판매가격에는 통계적으로 유의한 차이가 있나요?**
  * **답변:** **네, 유의한 차이가 있습니다.** `equal_var=True`로 수행한 독립표본 t검정 결과 t = 3.4249, p-value = 0.0015로 유의수준 0.05보다 작아 귀무가설을 기각하며, 주방 품질이 'Gd'인 주택의 평균 판매가격($191,916.60)이 'TA'인 주택($137,432.50)보다 통계적으로 유의하게 높습니다.

> 서로 다른 주택이라는 사실은 대응표본이 아니라는 근거입니다. 공간적 군집 등 관측치 간 의존성이 없는지는 수집 구조로 추가 확인해야 합니다. Shapiro-Wilk와 Levene의 p≥0.05는 가정을 기각할 증거가 부족하다는 의미입니다.


---

## 과제. 대응표본과 독립표본 상황 구분

### 문제 3-1. 같은 주택의 보수 전후 예상 판매가격 비교

#### 문제 설명

부동산 회사가 동일한 주택 10채에 대해 보수 전 예상 판매가격과 보수 후 예상 판매가격을 각각 산정했습니다. 같은 위치의 값은 동일한 주택의 전후 가격으로 서로 짝을 이룹니다.

```python
before_price = [145, 162, 178, 155, 190, 172, 168, 181, 159, 175]
after_price  = [154, 170, 185, 164, 201, 179, 176, 190, 168, 183]
```

단위는 천 달러입니다.

#### 요구사항

1. 두 배열의 길이가 같은지 확인하세요.
2. 보수 전후 평균을 계산하세요.
3. 독립표본 t검정과 대응표본 t검정 중 적절한 방법을 선택하고 이유를 설명하세요.
4. 대응표본 t검정의 정규성 가정은 개별 배열이 아니라 `after_price - before_price` 차이값에 적용된다는 점을 확인하세요.
5. 차이값에 Shapiro-Wilk 정규성 검정을 수행하세요.
6. 적절한 t검정을 실행하고 t통계량과 p-value를 출력하세요.
7. 보수 전후 평균 예상 판매가격에 유의한 차이가 있는지 결론을 작성하세요.

#### 해석 질문

**Q1.** 이 데이터는 독립표본인가요, 대응표본인가요?  
**Q2.** 대응표본 t검정에서는 무엇의 정규성을 확인해야 하나요?  
**Q3.** `stats.ttest_ind()`가 아니라 어떤 함수를 사용해야 하나요?  
**Q4.** 검정 결과 보수 전후 예상 판매가격에 유의한 차이가 있나요?

#### 제출 결과

- 표본 관계 판단과 근거
- 전후 평균과 차이값
- 차이값의 정규성 결과
- 대응표본 t검정 결과
- 결과 해석
- Q1~Q4 답변

In [4]:
# 같은 위치의 값은 같은 주택의 전후 가격이며 단위는 천 달러입니다.
before_price = np.array([145, 162, 178, 155, 190, 172, 168, 181, 159, 175])
after_price = np.array([154, 170, 185, 164, 201, 179, 176, 190, 168, 183])
assert len(before_price) == len(after_price)
diff = after_price - before_price
print("쌍 개수:", len(diff))
print("전 평균:", before_price.mean(), "후 평균:", after_price.mean())
print("차이값:", diff, "평균 차이:", diff.mean())
normality = stats.shapiro(diff)
print(f"차이값 Shapiro-Wilk: W={normality.statistic:.6f}, p={normality.pvalue:.6g}")
# H0: 보수 후 - 보수 전의 모집단 평균 차이 = 0; H1: 평균 차이 != 0
paired = stats.ttest_rel(after_price, before_price, alternative="two-sided")
print(f"대응표본 t검정: t={paired.statistic:.6f}, df={len(diff)-1}, p={paired.pvalue:.6g}")
assert np.isclose(paired.statistic, stats.ttest_1samp(diff, 0).statistic)
print("귀무가설 기각" if paired.pvalue < 0.05 else "귀무가설 기각 불가")


쌍 개수: 10
전 평균: 168.5 후 평균: 177.0
차이값: [ 9  8  7  9 11  7  8  9  9  8] 평균 차이: 8.5
차이값 Shapiro-Wilk: W=0.887127, p=0.157363
대응표본 t검정: t=22.807893, df=9, p=2.84201e-09
귀무가설 기각


### 과제 답변

- **Q1.** 동일한 주택 10채를 두 번 측정한 **대응표본**입니다. 길이가 같은 것뿐 아니라 각 위치가 동일 주택이라는 대응 관계가 핵심이며, 주택 간에는 독립성을 가정합니다.

- **Q2.** 개별 전·후 배열이 아니라 **후−전 차이값**의 정규성을 확인합니다. Shapiro-Wilk p=0.1574로 0.05에서 정규성을 기각하지 못하지만, 소표본에서 정규성이 증명된 것은 아닙니다.

- **Q3.** `stats.ttest_rel(after_price, before_price)`를 사용합니다. 대응을 유지한 차이값에 `ttest_1samp(diff, 0)`을 적용해도 같은 t값입니다.

- **Q4.** 전 평균 168.5, 후 평균 177.0, 평균 증가 8.5천 달러입니다. t=22.8079, p=2.842e-09이므로 평균 차이 0을 기각합니다. 이는 제시된 예상가격 자료의 차이이며 실제 거래가격에 대한 인과효과를 직접 입증하지는 않습니다.


---

## 실습 마무리

1. t통계량은 어떤 두 값을 비교하여 계산하나요?
2. 단일표본 t검정은 어떤 상황에서 사용하나요?
3. 독립표본과 대응표본은 데이터 수집 구조에서 어떤 차이가 있나요?
4. 독립표본 t검정에서 정규성·등분산성·독립성은 각각 어떻게 확인하나요?
5. 분산이 같다고 보기 어렵거나 확신할 수 없을 때 어떤 t검정을 사용할 수 있나요?
### 마무리 답변

1. t=(관측 평균−귀무가설의 기준 평균)/표준오차로, 평균 차이를 표본오차에 비추어 평가합니다.

2. 한 모집단의 평균을 특정 기준값과 비교하며 모분산을 모를 때 단일표본 t검정을 사용합니다.

3. 독립표본은 서로 대응하지 않는 별개 대상들, 대응표본은 같은 대상의 반복 측정 또는 사전에 짝지은 대상들입니다.

4. 정규성은 집단별 Shapiro-Wilk 및 분포 그림, 등분산성은 Levene 검정 등으로 점검합니다. 독립성은 검정 p값이 아니라 표집·수집 구조로 판단합니다. 가정 검정의 비유의성은 가정의 증명이 아닙니다.

5. 등분산을 가정하기 어렵다면 Welch t검정(`equal_var=False`)을 사용합니다. Welch도 관측치 간 독립성은 필요합니다.
